# Enzyme detection with ProtT5 and a neural network

This early run used 1,024-dimensional ProtT5 embeddings with an 80/20 split of the undersampled c100 dataset. Supply the original embedding file to run the experiment.

In [ ]:
import sys
from pathlib import Path

sys.path.append('..')

In [ ]:
from src.preprocessing import create_dataset_from_h5

embedding_file = Path('../data/embeddings/embeddings_prott5_exp_proteins_c100_flens25-75_usamp_enz.h5')
dataset = create_dataset_from_h5(embedding_file)

In [ ]:
from src.preprocessing import split_dataset

train_set, test_set = split_dataset(dataset=dataset, train_size=0.8)

In [ ]:
from src.train import train_model
from src.models import build_model, build_model_from_pretrained
from src.infer import infer

config = Path('../../results/task1/model_selection/crossval_runs/run_19/run_19_config.yaml')
model = build_model(config)
model_name = f'task1_first_model_{train_set.name}'

In [ ]:
import torch.optim as optim
import torch.nn as nn

o = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()
Path('../results/models').mkdir(parents=True, exist_ok=True)
model = train_model(model, model_name, train_set, optimizer=o, criterion=criterion, epochs=200, early_stopping=True, patience=15, show_training_curves=False)

In [ ]:
saved_model = build_model_from_pretrained(model_name)
infer(saved_model, test_set, metrics=True)

### Recorded test result

The original notebook recorded accuracy 0.96753 and MCC 0.93505. The later [ProtT5/PCA model](../../results/task1/model_selection/) used a different dataset and split.